# BƯỚC 2: CHUẨN HÓA ĐẶC TRƯNG (FEATURE SCALING & NORMALIZATION)

## Vì sao chuẩn hóa là bắt buộc với dữ liệu khách hàng?
Ba đặc trưng phân cụm có thang đo chênh lệch hàng nghìn lần:
- `Age`: `18 – 74` năm (`Std ≈ 11.7`).
- `Income`: `1,730 – 162,397` USD/năm (`Std ≈ 21,500`).
- `Total_Spending`: `5 – 2,525` USD (`Std ≈ 602`, lệch phải `Skew ≈ 0.86`).

Nếu không chuẩn hóa, khoảng cách Euclidean trong K-Means / Hierarchical / DBSCAN sẽ bị `Income` chi phối gần như tuyệt đối.

## Chiến lược đối chiếu & nguyên tắc chống Data Leakage
1. So sánh 4 trạng thái thang đo: `Gốc (Raw)`, `StandardScaler` ($Z = (x - \mu)/\sigma$), `MinMaxScaler` ($[0, 1]$) và `RobustScaler` (dựa trên Median & IQR).
2. **Chỉ `.fit()` trên tập Train (80%)**, sau đó `.transform()` trên Train và Holdout Test (20%).
3. Hậu kiểm tiêu chí hậu biến đổi (`Mean ≈ 0, Std ≈ 1` cho Standard; độ nhạy vượt biên `[0, 1]` trên Test của MinMax).
4. Đối chiếu thực nghiệm tác động của 3 scaler lên chất lượng K-Means trước khi chốt `StandardScaler`.

In [ ]:
# Import các module chuẩn hóa và nạp tập Train / Holdout Test đã làm sạch
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import NUMERICAL_FEATURES, PROCESSED_DATA_DIR
from src.evaluation.visualization import plot_scaling_comparison
from src.features.scaling import (
    FeatureScaler,
    compare_scalers,
    evaluate_scalers_on_clustering,
)

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
print(f"Đã nạp tập Train: {len(train_df):,} dòng | Tập Test: {len(test_df):,} dòng")

### 2.1 So sánh thống kê phân phối trên 4 thang đo (Raw, StandardScaler, MinMaxScaler, RobustScaler)
Bảng dưới bổ sung **Median**, **IQR** và **Skewness** để thấy rõ sự khác biệt về độ lệch và khả năng kháng ngoại lai giữa 3 bộ chuẩn hóa.

In [ ]:
bang_so_sanh = compare_scalers(train_df)
bang_so_sanh

### 2.2 Fit StandardScaler chỉ trên Train, kiểm tra tham số học được & hậu kiểm Train/Test
Đồng thời đối chiếu `MinMaxScaler` trên tập Holdout Test để quan sát hiện tượng mẫu Test vượt khỏi đoạn `[0, 1]` khi có giá trị ngoài cận Min/Max của Train.

In [ ]:
scaler = FeatureScaler(method="standard")
ma_tran_train = scaler.fit_transform(train_df)
ma_tran_test = scaler.transform(test_df)

print("--- Tham số StandardScaler học chỉ từ tập Train ---")
print(scaler.get_learned_params().to_string(index=False))

minmax_doi_chieu = FeatureScaler(method="minmax").fit(train_df)
bang_hau_kiem = pd.concat(
    [
        scaler.verify_scaled_matrix(ma_tran_train, is_train=True),
        scaler.verify_scaled_matrix(ma_tran_test, is_train=False),
        minmax_doi_chieu.verify_scaled_matrix(minmax_doi_chieu.transform(test_df), is_train=False),
    ],
    ignore_index=True,
)
print("--- Hậu kiểm hậu điều kiện trên Train và Holdout Test ---")
bang_hau_kiem

### 2.3 Khảo sát tác động của 3 bộ chuẩn hóa lên chất lượng phân cụm K-Means
Giữ cố định tập Train/Test, 3 đặc trưng số và thuật toán `K-Means (k-means++)` để đánh giá định lượng sự đánh đổi giữa `StandardScaler`, `MinMaxScaler` và `RobustScaler`.

In [ ]:
bang_anh_huong = evaluate_scalers_on_clustering(train_df, test_df=test_df)
bang_anh_huong

### 2.4 Trực quan hóa phân phối 4 trạng thái & kiểm tra tính khả nghịch (`inverse_transform`)
`StandardScaler` được chọn làm bộ chuẩn hóa chính vì cân bằng cả 3 đặc trưng (`Mean=0, Std=1`), đạt **Davies-Bouldin tốt hơn**, bảo toàn hình thái phân phối gốc cho K-Means/Ward và không phụ thuộc cận Min/Max cứng khi dự đoán dữ liệu mới.

In [ ]:
plot_scaling_comparison(train_df, ma_tran_train)

khoi_phuc = scaler.inverse_transform(ma_tran_test[:3])
print("--- Kiểm tra khả năng quy đổi ngược inverse_transform (3 mẫu Test đầu) ---")
print("Giá trị gốc :\n", test_df[NUMERICAL_FEATURES].head(3).to_numpy())
print("Khôi phục lại:\n", khoi_phuc.round(4))